# Phishing URL Detection Model Training

This notebook trains an improved ML model for phishing URL detection with focus on **high precision** to reduce false positives on legitimate sites.

**Key Features:**
- No hardcoded whitelist - model learns to identify legitimate sites accurately
- Multiple model comparison (Random Forest, Gradient Boosting, AdaBoost, XGBoost, Ensemble)
- Optimized hyperparameters for precision
- Uses combined Mendeley + UCI datasets

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# Try to import XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("✓ XGBoost available")
except:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not available. Will use other models.")

print("Libraries imported successfully!")

## Step 1: Load Datasets

Load and examine the Mendeley and UCI phishing datasets.

In [ ]:
# Load datasets
print("Loading datasets...")
mendeley_df = pd.read_csv('Model Training DataSets/Mendeley Dataset.csv')
uci_df = pd.read_csv('Model Training DataSets/UCI PhiUSIIL_Phishing_URL UCI DataSet.csv')

print(f"\nMendeley dataset shape: {mendeley_df.shape}")
print(f"UCI dataset shape: {uci_df.shape}")

# Display first few rows
print("\nMendeley dataset columns:", mendeley_df.columns.tolist()[:10], "...")
print("UCI dataset columns:", uci_df.columns.tolist()[:10], "...")

## Step 4: Combine Datasets and Split

Combine both datasets and split into training and testing sets.

In [ ]:
# Combine datasets
X_combined = pd.concat([mendeley_X, uci_X], ignore_index=True)
y_combined = pd.concat([mendeley_y, uci_df['label']], ignore_index=True)

print(f"Combined dataset shape: {X_combined.shape}")
print(f"\nLabel distribution:")
print(y_combined.value_counts())

# Handle NaN values
X_combined = X_combined.fillna(0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Step 5: Train Multiple Models

Train multiple models optimized for precision to reduce false positives.

In [ ]:
# Initialize results dictionary
results = {}

# Model 1: Random Forest (Optimized for precision)
print("1. Training Random Forest (Precision-focused)...")
rf_model = RandomForestClassifier(
    n_estimators=600,
    max_depth=35,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    class_weight={0: 1.0, 1: 0.85}  # Slightly favor legitimate to reduce false positives
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

results['RandomForest'] = {
    'model': rf_model,
    'accuracy': accuracy_score(y_test, rf_pred),
    'roc_auc': roc_auc_score(y_test, rf_pred_proba),
    'precision': precision_score(y_test, rf_pred),
    'recall': recall_score(y_test, rf_pred),
    'f1': f1_score(y_test, rf_pred)
}
print(f"   ✓ Accuracy: {results['RandomForest']['accuracy']:.4f}")
print(f"   ✓ Precision: {results['RandomForest']['precision']:.4f}")
print(f"   ✓ Recall: {results['RandomForest']['recall']:.4f}")
print(f"   ✓ F1-Score: {results['RandomForest']['f1']:.4f}\n")

In [ ]:
# Model 2: Gradient Boosting (Optimized for precision)
print("2. Training Gradient Boosting (Precision-focused)...")
gb_model = GradientBoostingClassifier(
    n_estimators=500,
    max_depth=18,
    learning_rate=0.02,
    subsample=0.85,
    max_features='sqrt',
    random_state=42
)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_pred_proba = gb_model.predict_proba(X_test)[:, 1]

results['GradientBoosting'] = {
    'model': gb_model,
    'accuracy': accuracy_score(y_test, gb_pred),
    'roc_auc': roc_auc_score(y_test, gb_pred_proba),
    'precision': precision_score(y_test, gb_pred),
    'recall': recall_score(y_test, gb_pred),
    'f1': f1_score(y_test, gb_pred)
}
print(f"   ✓ Accuracy: {results['GradientBoosting']['accuracy']:.4f}")
print(f"   ✓ Precision: {results['GradientBoosting']['precision']:.4f}")
print(f"   ✓ Recall: {results['GradientBoosting']['recall']:.4f}")
print(f"   ✓ F1-Score: {results['GradientBoosting']['f1']:.4f}\n")

In [ ]:
# Model 3: AdaBoost
print("3. Training AdaBoost...")
ada_model = AdaBoostClassifier(
    n_estimators=300,
    learning_rate=0.1,
    random_state=42
)
ada_model.fit(X_train, y_train)
ada_pred = ada_model.predict(X_test)
ada_pred_proba = ada_model.predict_proba(X_test)[:, 1]

results['AdaBoost'] = {
    'model': ada_model,
    'accuracy': accuracy_score(y_test, ada_pred),
    'roc_auc': roc_auc_score(y_test, ada_pred_proba),
    'precision': precision_score(y_test, ada_pred),
    'recall': recall_score(y_test, ada_pred),
    'f1': f1_score(y_test, ada_pred)
}
print(f"   ✓ Accuracy: {results['AdaBoost']['accuracy']:.4f}")
print(f"   ✓ Precision: {results['AdaBoost']['precision']:.4f}")
print(f"   ✓ Recall: {results['AdaBoost']['recall']:.4f}")
print(f"   ✓ F1-Score: {results['AdaBoost']['f1']:.4f}\n")

In [ ]:
# Model 4: XGBoost (if available)
if XGBOOST_AVAILABLE:
    print("4. Training XGBoost (Precision-focused)...")
    try:
        xgb_model = xgb.XGBClassifier(
            n_estimators=500,
            max_depth=18,
            learning_rate=0.02,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            eval_metric='logloss',
            use_label_encoder=False,
            scale_pos_weight=0.85  # Reduce false positives
        )
        xgb_model.fit(X_train, y_train)
        xgb_pred = xgb_model.predict(X_test)
        xgb_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

        results['XGBoost'] = {
            'model': xgb_model,
            'accuracy': accuracy_score(y_test, xgb_pred),
            'roc_auc': roc_auc_score(y_test, xgb_pred_proba),
            'precision': precision_score(y_test, xgb_pred),
            'recall': recall_score(y_test, xgb_pred),
            'f1': f1_score(y_test, xgb_pred)
        }
        print(f"   ✓ Accuracy: {results['XGBoost']['accuracy']:.4f}")
        print(f"   ✓ Precision: {results['XGBoost']['precision']:.4f}")
        print(f"   ✓ Recall: {results['XGBoost']['recall']:.4f}")
        print(f"   ✓ F1-Score: {results['XGBoost']['f1']:.4f}\n")
    except Exception as e:
        print(f"   ✗ XGBoost training failed: {e}\n")
else:
    print("4. XGBoost not available, skipping...\n")

In [ ]:
# Model 5: Ensemble (Voting Classifier)
print("5. Training Ensemble Model (Voting Classifier)...")
ensemble_models = []
if 'RandomForest' in results:
    ensemble_models.append(('rf', results['RandomForest']['model']))
if 'GradientBoosting' in results:
    ensemble_models.append(('gb', results['GradientBoosting']['model']))
if 'XGBoost' in results:
    ensemble_models.append(('xgb', results['XGBoost']['model']))

if len(ensemble_models) >= 2:
    ensemble = VotingClassifier(estimators=ensemble_models, voting='soft')
    ensemble.fit(X_train, y_train)
    ensemble_pred = ensemble.predict(X_test)
    ensemble_pred_proba = ensemble.predict_proba(X_test)[:, 1]

    results['Ensemble'] = {
        'model': ensemble,
        'accuracy': accuracy_score(y_test, ensemble_pred),
        'roc_auc': roc_auc_score(y_test, ensemble_pred_proba),
        'precision': precision_score(y_test, ensemble_pred),
        'recall': recall_score(y_test, ensemble_pred),
        'f1': f1_score(y_test, ensemble_pred)
    }
    print(f"   ✓ Accuracy: {results['Ensemble']['accuracy']:.4f}")
    print(f"   ✓ Precision: {results['Ensemble']['precision']:.4f}")
    print(f"   ✓ Recall: {results['Ensemble']['recall']:.4f}")
    print(f"   ✓ F1-Score: {results['Ensemble']['f1']:.4f}\n")
else:
    print("   ⚠️  Need at least 2 models for ensemble, skipping...\n")

## Step 6: Model Comparison and Selection

Compare all models and select the best one based on precision (to reduce false positives).

In [ ]:
# Select best model based on precision (reduce false positives), then accuracy and F1-score
best_model_name = max(results.keys(), key=lambda k: (
    results[k]['precision'],  # Highest precision first
    results[k]['accuracy'],    # Then accuracy
    results[k]['f1']          # Then F1-score
))
best_model = results[best_model_name]['model']
best_results = results[best_model_name]

# Display comparison
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[k]['accuracy'] for k in results.keys()],
    'Precision': [results[k]['precision'] for k in results.keys()],
    'Recall': [results[k]['recall'] for k in results.keys()],
    'F1-Score': [results[k]['f1'] for k in results.keys()],
    'ROC-AUC': [results[k]['roc_auc'] for k in results.keys()]
}).sort_values('Precision', ascending=False)

comparison_df['Best'] = comparison_df['Model'] == best_model_name
print("Model Comparison:")
print(comparison_df.to_string(index=False))

print(f"\n{'='*70}")
print(f"✓ Best Model: {best_model_name}")
print(f"  Accuracy: {best_results['accuracy']:.4f} ({best_results['accuracy']*100:.2f}%)")
print(f"  Precision: {best_results['precision']:.4f} ({best_results['precision']*100:.2f}%)")
print(f"  Recall: {best_results['recall']:.4f} ({best_results['recall']*100:.2f}%)")
print(f"  F1-Score: {best_results['f1']:.4f}")
print(f"  ROC-AUC: {best_results['roc_auc']:.4f}")
print(f"{'='*70}")

In [ ]:
# Detailed classification report
print("\nDetailed Classification Report:")
best_pred = best_model.predict(X_test)
print(classification_report(y_test, best_pred, target_names=['Legitimate', 'Phishing']))

## Step 7: Save Model

Save the best model, features, and metadata.

In [ ]:
# Save the best model
print("Saving model...")
joblib.dump(best_model, 'phishing_detection_model.pkl')
joblib.dump(mendeley_features, 'model_features.pkl')

model_info = {
    'model_type': best_model_name,
    'accuracy': float(best_results['accuracy']),
    'roc_auc': float(best_results['roc_auc']),
    'precision': float(best_results['precision']),
    'recall': float(best_results['recall']),
    'f1_score': float(best_results['f1']),
    'n_features': len(mendeley_features),
    'n_train_samples': len(X_train),
    'n_test_samples': len(X_test),
    'features': mendeley_features,
    'all_model_results': {k: {m: float(v) for m, v in metrics.items() if m != 'model'} for k, metrics in results.items()},
    'note': 'Trained with focus on precision to reduce false positives. No whitelist required - model learns to identify legitimate sites accurately.'
}

with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("✓ Model saved as 'phishing_detection_model.pkl'")
print("✓ Features saved as 'model_features.pkl'")
print("✓ Model info saved as 'model_info.json'")

## Feature Importance

Display the top 20 most important features for the best model.

In [ ]:
# Feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': mendeley_features,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Top 20 Most Important Features:")
    print(feature_importance.head(20).to_string(index=False))
    
    # Optional: Visualize feature importance
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 8))
        top_features = feature_importance.head(20)
        plt.barh(range(len(top_features)), top_features['importance'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Importance')
        plt.title('Top 20 Feature Importances')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
    except:
        print("(Matplotlib not available for visualization)")